**Chirag Bansal - 102303700 - assignment 4**

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

book_titles=[]
book_prices=[]
stock_status=[]
ratings=[]

page_url="https://books.toscrape.com/catalogue/page-1.html"
root_url="https://books.toscrape.com/catalogue/"

while page_url:
    response=requests.get(page_url).text
    parsed_html=BeautifulSoup(response,"html.parser")

    all_books=parsed_html.find_all("article",class_="product_pod")

    for item in all_books:
        book_title=item.h3.a["title"]
        book_titles.append(book_title)

        book_price=item.find("p",class_="price_color").text
        book_prices.append(book_price)

        stock_text=item.find("p",class_="instock availability").text.strip()
        stock_status.append(stock_text)

        rating_class=item.find("p",class_="star-rating")["class"]
        rating_value=[c for c in rating_class if c!="star-rating"][0]
        ratings.append(rating_value)

    next_link=parsed_html.find("li",class_="next")
    if next_link and next_link.a:
        next_relative=next_link.a["href"]
        page_url=urljoin(root_url,next_relative)
    else:
        page_url=None

data_frame=pd.DataFrame({
    "Title":book_titles,
    "Price":book_prices,
    "Availability":stock_status,
    "Star Rating":ratings
})

print(data_frame.head(10))
print("\n")
print(data_frame.tail(10))
print("\n")
print(data_frame.describe())

data_frame.to_csv("books.csv",index=False)

                                               Title    Price Availability  \
0                               A Light in the Attic  Â£51.77     In stock   
1                                 Tipping the Velvet  Â£53.74     In stock   
2                                         Soumission  Â£50.10     In stock   
3                                      Sharp Objects  Â£47.82     In stock   
4              Sapiens: A Brief History of Humankind  Â£54.23     In stock   
5                                    The Requiem Red  Â£22.65     In stock   
6  The Dirty Little Secrets of Getting Your Dream...  Â£33.34     In stock   
7  The Coming Woman: A Novel Based on the Life of...  Â£17.93     In stock   
8  The Boys in the Boat: Nine Americans and Their...  Â£22.60     In stock   
9                                    The Black Maria  Â£52.15     In stock   

  Star Rating  
0       Three  
1         One  
2         One  
3        Four  
4        Five  
5         One  
6        Four  
7       Three

In [7]:
!pip install selenium

!apt-get update
!apt install chromium-chromedriver

import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

chrome_options=webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

browser=webdriver.Chrome(options=chrome_options)

web_url="https://www.imdb.com/chart/top/"

browser.get(web_url)

time.sleep(5)

prev_height=browser.execute_script("return document.body.scrollHeight")

print("Scrolling down to load all movies...")
for i in range(10):
    browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1)

print("Finished loading the page.")

html_source=browser.page_source
parsed_page=BeautifulSoup(html_source,"html.parser")

browser.quit()

film_items=parsed_page.find_all('li',class_='ipc-metadata-list-summary-item')

print(f"Found {len(film_items)} movie items")

position=[]
film_titles=[]
release_years=[]
imdb_scores=[]

for film in film_items:
    heading=film.find('h3',class_='ipc-title__text')
    if heading:
        complete_text=heading.text.strip()
        if '.' in complete_text:
            pos_num,film_name=complete_text.split('.',1)
            position.append(pos_num.strip())
            film_titles.append(film_name.strip())
        else:
            position.append("N/A")
            film_titles.append(complete_text)
    else:
        position.append("N/A")
        film_titles.append("N/A")

    meta_section=film.find('div',class_='cli-title-metadata')
    if meta_section:
        meta_items=meta_section.find_all('span',class_='cli-title-metadata-item')
        if meta_items:
            release_years.append(meta_items[0].text.strip())
        else:
            release_years.append("N/A")
    else:
        release_years.append("N/A")

    score_div=film.find('span',class_='ipc-rating-star--rating')
    if score_div:
        imdb_scores.append(score_div.text.strip())
    else:
        imdb_scores.append("N/A")

output_df=pd.DataFrame({
    "Rank":position,
    "Title":film_titles,
    "Year":release_years,
    "IMDb Rating":imdb_scores
})

print(f"\nSuccessfully scraped {len(output_df)} movies.")
print(output_df.head(20))
print(output_df.tail(20))

output_df.to_csv("imdb_top_250.csv",index=False)

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [8]:
#QUESTION 3
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://www.timeanddate.com/weather/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'html.parser')

weather_data = []

main_table = soup.find('table', class_='zebra fw tb-theme')

if main_table:
    rows = main_table.find_all('tr')

    for row in rows[1:]:
        cells = row.find_all('td')

        for i in range(0, len(cells), 4):
            if i + 3 < len(cells):
                city_cell = cells[i]
                weather_cell = cells[i+2]
                temp_cell = cells[i+3]

                city_link = city_cell.find('a', href=True)
                city_name = city_link.text.strip() if city_link else 'N/A'

                img_tag = weather_cell.find('img')
                if img_tag and img_tag.has_attr('alt'):
                    weather_condition = img_tag['alt'].strip()
                else:
                    weather_condition = 'N/A'

                # Temperature
                temperature = temp_cell.text.strip() if temp_cell else 'N/A'

                # Append data
                weather_data.append([city_name, temperature, weather_condition])

df = pd.DataFrame(weather_data, columns=['City Name', 'Temperature', 'Weather Condition'])

print(f"Total cities extracted: {len(df)}")
print(df.head(10))
df.to_csv('weather_data.csv', index=False)


Total cities extracted: 141
     City Name Temperature                     Weather Condition
0        Accra       77 °F                          Clear. Warm.
1       Dublin       43 °F           Passing clouds. Quite cool.
2      Nairobi       64 °F                  Broken clouds. Mild.
3  Addis Ababa       59 °F  Scattered clouds. Refreshingly cool.
4     Edmonton       50 °F                 Passing clouds. Cool.
5       Nassau       81 °F                 Passing clouds. Warm.
6     Adelaide       72 °F                                 Mild.
7    Frankfurt       46 °F                 Passing clouds. Cool.
8    New Delhi       68 °F                            Fog. Mild.
9      Algiers       73 °F                          Clear. Mild.
